**Install Dependencies**

In [ ]:
!pip install -q transformers accelerate qwen-vl-utils[decord] av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 101.5 MB/s eta 0:00:00


**Load Model**

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

**Set Up Conversation with Video**

In [ ]:

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": "my_video.mp4",
                "max_pixels": 360 * 420,
                "fps": 1.0,
            },
            {"type": "text", "text": "Describe what happens in this video."},
        ],
    }
]


def ask(question, messages):
    messages.append({"role": "user", "content": [{"type": "text", "text": question}]})

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to(model.device)

    output_ids = model.generate(**inputs, max_new_tokens=512)
    response = processor.batch_decode(
        output_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )[0]

    messages.append({"role": "assistant", "content": [{"type": "text", "text": response}]})
    return response

print("Qwen:", ask("Describe what happens in this video.", messages))


print("\nNow ask follow-up questions (type 'quit' to stop).")
while True:
    q = input("You: ")
    if q.lower() in ("quit", "exit"):
        break
    response = ask(q, messages)
    print("Qwen:", response)

qwen-vl-utils using torchcodec to read video.


Qwen: The video starts with a view from inside a car driving on a highway. The sky is partly cloudy, and the road is wide with multiple lanes. Several cars are visible ahead, including a white SUV and a red car. The car continues to drive forward, passing other vehicles and approaching an intersection. As it approaches the intersection, the camera angle shifts slightly, showing more of the surrounding area, including other cars and traffic signs. The car then turns right at the intersection, continuing down another highway. The sky remains partly cloudy throughout the video.

Now ask follow-up questions (type 'quit' to stop).
You: is there traffic?
Qwen: Yes, there is traffic in the video. The car is driving on a highway with several other vehicles, including a white SUV and a red car. The video shows the car passing these vehicles as it drives forward.
You: can you see any bike?
Qwen: No, I cannot see any bikes in the video. The video primarily shows a car driving on a highway with ot

**Live Webcam Inference**

In [ ]:
from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js
from base64 import b64decode
import time

def start_camera():
    js = Javascript('''
        async function startCamera() {
            if (window._stream) return "already running";
            const video = document.createElement('video');
            video.id = 'liveVideoPreview';
            video.style.width = '320px';
            video.style.borderRadius = '8px';
            video.muted = true;
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            document.body.appendChild(video);
            video.srcObject = stream;
            await video.play();
            while (video.videoWidth === 0) {
                await new Promise(r => setTimeout(r, 100));
            }
            window._stream = stream;
            window._video = video;
            return "ready";
        }
    ''')
    display(js)
    return eval_js('startCamera()')

def grab_frame(filename='webcam.jpg', quality=0.8):
    js = Javascript('''
        async function grabFrame(quality) {
            const video = window._video;
            if (!video || video.videoWidth === 0) {
                throw new Error("Camera not ready yet");
            }
            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    display(js)
    data = eval_js('grabFrame({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

def stop_camera():
    js = Javascript('''
        function stopCamera() {
            if (window._stream) {
                window._stream.getTracks().forEach(t => t.stop());
                window._stream = null;
            }
            if (window._video) {
                window._video.remove();
                window._video = null;
            }
        }
    ''')
    display(js)
    eval_js('stopCamera()')

def ask_about_photo(photo_path, question="What am I holding in my hand?"):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": photo_path},
                {"type": "text", "text": question},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to(model.device)
    output_ids = model.generate(**inputs, max_new_tokens=128)
    return processor.batch_decode(
        output_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )[0]

# --- Run everything ---
print("Starting camera...")
start_camera()
print("Camera preview should appear above (live feed). Answers will update below.\n")

handle = display("Waiting for first answer...", display_id=True)

try:
    while True:
        photo_path = grab_frame()
        answer = ask_about_photo(photo_path)
        handle.update(f"Qwen: {answer}")
        time.sleep(2)
except KeyboardInterrupt:
    print("\nStopped — releasing camera...")
finally:
    stop_camera()
    print("Camera released.")

Starting camera...


<IPython.core.display.Javascript object>

Camera preview should appear above (live feed). Answers will update below.



'Qwen: You are holding a tube of foundation and a makeup palette with various shades.'

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Stopped — releasing camera...


<IPython.core.display.Javascript object>

Camera released.


**Live Speech Inference**





In [ ]:
!pip install -q openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 16.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from IPython.display import display, Javascript, Audio
from google.colab.output import eval_js
from base64 import b64decode
import whisper
import time

print("Loading Whisper...")
whisper_model = whisper.load_model("base")
print("Whisper loaded.\n")

# --- Camera functions ---
def start_camera():
    js = Javascript('''
        async function startCamera() {
            if (window._stream) return "already running";
            const video = document.createElement('video');
            video.style.width = '320px';
            video.style.borderRadius = '8px';
            video.muted = true;
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            document.body.appendChild(video);
            video.srcObject = stream;
            await video.play();
            while (video.videoWidth === 0) {
                await new Promise(r => setTimeout(r, 100));
            }
            window._stream = stream;
            window._video = video;
            return "ready";
        }
    ''')
    display(js)
    return eval_js('startCamera()')

def grab_frame(filename='webcam.jpg', quality=0.8):
    js = Javascript('''
        async function grabFrame(quality) {
            const video = window._video;
            if (!video || video.videoWidth === 0) throw new Error("Camera not ready yet");
            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    display(js)
    data = eval_js('grabFrame({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

def stop_camera():
    js = Javascript('''
        function stopCamera() {
            if (window._stream) { window._stream.getTracks().forEach(t => t.stop()); window._stream = null; }
            if (window._video) { window._video.remove(); window._video = null; }
        }
    ''')
    display(js)
    eval_js('stopCamera()')

# --- Audio recording (raw capture, not SpeechRecognition) ---
def record_audio(filename='question.wav', seconds=4):
    js = Javascript(f'''
        async function recordAudio() {{
            const stream = await navigator.mediaDevices.getUserMedia({{ audio: true }});
            const recorder = new MediaRecorder(stream);
            const chunks = [];
            recorder.ondataavailable = (e) => chunks.push(e.data);
            const stopped = new Promise((resolve) => recorder.onstop = resolve);

            recorder.start();
            await new Promise(r => setTimeout(r, {seconds * 1000}));
            recorder.stop();
            await stopped;

            stream.getTracks().forEach(t => t.stop());
            const blob = new Blob(chunks, {{ type: 'audio/webm' }});
            const buffer = await blob.arrayBuffer();
            const bytes = new Uint8Array(buffer);
            let binary = '';
            for (let i = 0; i < bytes.byteLength; i++) binary += String.fromCharCode(bytes[i]);
            return btoa(binary);
        }}
    ''')
    display(js)
    b64_audio = eval_js('recordAudio()')
    binary = b64decode(b64_audio)
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

def transcribe_audio(audio_path):
    result = whisper_model.transcribe(audio_path, fp16=False)
    return result['text'].strip()

def speak(text):
    js = Javascript(f'''
        const utterance = new SpeechSynthesisUtterance({text!r});
        speechSynthesis.speak(utterance);
    ''')
    display(js)

# --- Qwen model call ---
def ask_about_photo(photo_path, question):
    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": photo_path},
            {"type": "text", "text": question},
        ]}
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(model.device)
    output_ids = model.generate(**inputs, max_new_tokens=128)
    return processor.batch_decode(output_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]

# --- Run everything ---
print("Starting camera...")
start_camera()
print("Camera ready.\n")
print("Loop: records 4s of audio -> transcribes -> asks Qwen -> speaks answer. Stop (⏹) to end.\n")

try:
    while True:
        print("\n🎤 Recording... (speak your question now, 4 seconds)")
        audio_path = record_audio(seconds=4)
        question = transcribe_audio(audio_path)
        if question:
            print(f"You asked: {question}")
            photo_path = grab_frame()
            answer = ask_about_photo(photo_path, question)
            print(f"Qwen: {answer}")
            speak(answer)
        else:
            print("Didn't catch anything, try again.")
        time.sleep(1)
except KeyboardInterrupt:
    print("\nStopped.")
finally:
    stop_camera()
    print("Camera released.")

Loading Whisper...
Whisper loaded.

Starting camera...


<IPython.core.display.Javascript object>

Camera ready.

Loop: records 4s of audio -> transcribes -> asks Qwen -> speaks answer. Stop (⏹) to end.


🎤 Recording... (speak your question now, 4 seconds)


<IPython.core.display.Javascript object>

You asked: Hello


...

You asked: He's getting a bow in-


<IPython.core.display.Javascript object>


Stopped.


<IPython.core.display.Javascript object>

Camera released.
